# Mudsplash

> **Task name:** `Mudsplash`

**Track: Attention** | Does salient content pull attention away from detecting changes?

In visual cognition, "mudsplash" experiments show that attention-grabbing stimuli
appearing simultaneously with a change cause people to miss the change entirely.
This benchmark tests whether LLMs are similarly vulnerable to attentional capture.

Three disruptor types test different capture mechanisms:

- **Neutral** — bland, factual filler text (baseline)
- **Emotional** — shocking, dramatic, attention-grabbing content
- **Task-relevant** — content about a similar topic that might confuse the model

**Metric:** Attentional capture effect — accuracy drop from neutral to emotional/task-relevant disruptors.

In [ ]:
# Last updated: 2026-03-31 04:59 UTC
import kaggle_benchmarks as kbench
import pandas as pd
import numpy as np
import json
import re
import random

print(f"Available models: {list(kbench.llms.keys())}")

In [ ]:
def strip_thinking(text: str) -> str:
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()

def normalize(s: str) -> str:
    return re.sub(r"\s+", " ", s.lower().strip())

In [ ]:
# --- Fictional passages with controlled changes ---

PASSAGES = [
    {
        "id": "volcano",
        "original": "Mount Pyrethon in the Valderi archipelago last erupted in 1847, releasing 3.2 cubic kilometers of ash into the stratosphere. Geologist Dr. Hana Mertovic's monitoring stations detect an average of 47 micro-tremors per month. The volcanic soil supports a unique ecosystem of 156 endemic plant species found nowhere else. Local authorities maintain an exclusion zone of 8 kilometers around the summit caldera.",
        "minor_change": "Mount Pyrethon in the Valderi archipelago last erupted in 1847, releasing 3.2 cubic kilometers of ash into the stratosphere. Geologist Dr. Hana Mertovic's monitoring stations detect an average of 23 micro-tremors per month. The volcanic soil supports a unique ecosystem of 156 endemic plant species found nowhere else. Local authorities maintain an exclusion zone of 8 kilometers around the summit caldera.",
        "minor_detail": "47 micro-tremors changed to 23 micro-tremors",
        "major_change": "Mount Pyrethon in the Valderi archipelago last erupted in 1847, releasing 3.2 cubic kilometers of ash into the stratosphere. Geologist Dr. Hana Mertovic's monitoring stations detect an average of 47 micro-tremors per month. The volcanic soil is completely barren and unable to support any form of plant life due to toxic mineral deposits. Local authorities maintain an exclusion zone of 8 kilometers around the summit caldera.",
        "major_detail": "unique ecosystem of 156 endemic plant species changed to completely barren and unable to support any plant life",
    },
    {
        "id": "submarine",
        "original": "The research submarine Abyssal-7 completed a 94-day mission mapping the Kermadec Trench floor. Captain Yuki Tanaka's crew of 12 scientists collected 340 sediment core samples from depths averaging 9,400 meters. Sonar imaging revealed 28 previously unknown hydrothermal vent clusters. The expedition's biological sampling yielded tissue from 73 species, including a translucent octopus with copper-based blood pigment.",
        "minor_change": "The research submarine Abyssal-7 completed a 94-day mission mapping the Kermadec Trench floor. Captain Yuki Tanaka's crew of 12 scientists collected 340 sediment core samples from depths averaging 9,400 meters. Sonar imaging revealed 14 previously unknown hydrothermal vent clusters. The expedition's biological sampling yielded tissue from 73 species, including a translucent octopus with copper-based blood pigment.",
        "minor_detail": "28 hydrothermal vent clusters changed to 14 hydrothermal vent clusters",
        "major_change": "The research submarine Abyssal-7 completed a 94-day mission mapping the Kermadec Trench floor. Captain Yuki Tanaka's crew of 12 scientists collected 340 sediment core samples from depths averaging 9,400 meters. Sonar imaging revealed 28 previously unknown hydrothermal vent clusters. The expedition's biological sampling found no living organisms whatsoever, confirming the trench floor is entirely devoid of life.",
        "major_detail": "yielded tissue from 73 species including translucent octopus changed to found no living organisms whatsoever",
    },
    {
        "id": "telescope",
        "original": "The Meridian Array radio telescope in the Chilean Andes comprises 64 dish antennas spanning a 12-kilometer baseline. Director Prof. Amara Osei's team detected a repeating fast radio burst designated FRB-2024-MR7 with a period of 16.35 days. Signal analysis suggests the source lies 2.4 billion light-years away in a dwarf galaxy with unusually high metallicity. The array's angular resolution of 0.001 arcseconds makes it the most precise radio instrument in the southern hemisphere.",
        "minor_change": "The Meridian Array radio telescope in the Chilean Andes comprises 64 dish antennas spanning a 12-kilometer baseline. Director Prof. Amara Osei's team detected a repeating fast radio burst designated FRB-2024-MR7 with a period of 16.35 days. Signal analysis suggests the source lies 5.1 billion light-years away in a dwarf galaxy with unusually high metallicity. The array's angular resolution of 0.001 arcseconds makes it the most precise radio instrument in the southern hemisphere.",
        "minor_detail": "2.4 billion light-years changed to 5.1 billion light-years",
        "major_change": "The Meridian Array radio telescope in the Chilean Andes comprises 64 dish antennas spanning a 12-kilometer baseline. Director Prof. Amara Osei's team detected a repeating fast radio burst designated FRB-2024-MR7 with a period of 16.35 days. Signal analysis confirms the burst is artificial in origin and represents deliberate communication from an extraterrestrial civilization. The array's angular resolution of 0.001 arcseconds makes it the most precise radio instrument in the southern hemisphere.",
        "major_detail": "natural source 2.4 billion light-years away in dwarf galaxy changed to artificial origin representing extraterrestrial communication",
    },
    {
        "id": "glacier",
        "original": "The Nordvik Glacier in eastern Greenland retreated 1.3 kilometers between 2020 and 2025, according to glaciologist Dr. Sven Halvorsen. Ice core analysis reveals 847 distinct annual layers dating back to 1178 CE. Embedded air bubbles show atmospheric CO2 levels have risen 43 percent since the deepest layers were deposited. The glacier feeds the Nordvik River, which provides hydroelectric power to approximately 120,000 residents.",
        "minor_change": "The Nordvik Glacier in eastern Greenland retreated 1.3 kilometers between 2020 and 2025, according to glaciologist Dr. Sven Halvorsen. Ice core analysis reveals 847 distinct annual layers dating back to 1178 CE. Embedded air bubbles show atmospheric CO2 levels have risen 43 percent since the deepest layers were deposited. The glacier feeds the Nordvik River, which provides hydroelectric power to approximately 85,000 residents.",
        "minor_detail": "120,000 residents changed to 85,000 residents",
        "major_change": "The Nordvik Glacier in eastern Greenland retreated 1.3 kilometers between 2020 and 2025, according to glaciologist Dr. Sven Halvorsen. Ice core analysis reveals 847 distinct annual layers dating back to 1178 CE. Embedded air bubbles show atmospheric CO2 levels have remained completely stable with zero change since the deepest layers were deposited. The glacier feeds the Nordvik River, which provides hydroelectric power to approximately 120,000 residents.",
        "major_detail": "CO2 risen 43 percent changed to CO2 remained completely stable with zero change",
    },
    {
        "id": "vaccine",
        "original": "Phase III trials of the RNV-4200 malaria vaccine enrolled 8,400 participants across 23 clinical sites in sub-Saharan Africa. Lead researcher Dr. Fatou Diallo reported 78 percent efficacy against Plasmodium falciparum infection over 18 months. Side effects were limited to mild injection site reactions in 12 percent of subjects. The WHO estimates deployment could prevent 240,000 deaths annually in children under five.",
        "minor_change": "Phase III trials of the RNV-4200 malaria vaccine enrolled 8,400 participants across 23 clinical sites in sub-Saharan Africa. Lead researcher Dr. Fatou Diallo reported 78 percent efficacy against Plasmodium falciparum infection over 18 months. Side effects were limited to mild injection site reactions in 12 percent of subjects. The WHO estimates deployment could prevent 160,000 deaths annually in children under five.",
        "minor_detail": "240,000 deaths changed to 160,000 deaths",
        "major_change": "Phase III trials of the RNV-4200 malaria vaccine enrolled 8,400 participants across 23 clinical sites in sub-Saharan Africa. Lead researcher Dr. Fatou Diallo reported 78 percent efficacy against Plasmodium falciparum infection over 18 months. Side effects included severe allergic reactions in over half of all subjects requiring emergency hospitalization. The WHO estimates deployment could prevent 240,000 deaths annually in children under five.",
        "major_detail": "mild injection site reactions in 12 percent changed to severe allergic reactions in over half requiring hospitalization",
    },
]

# Three disruptor types: neutral, emotional, task-relevant
DISRUPTORS = {
    "neutral": [
        "The standardization of shipping container dimensions in 1956 reduced cargo loading times by approximately 75 percent. International Maritime Organization regulations now specify eight standard container sizes for intermodal transport.",
        "Municipal composting programs in Scandinavian countries divert an average of 62 percent of organic waste from landfills. The resulting compost is distributed to local agricultural cooperatives at subsidized rates.",
        "The Library of Congress classification system assigns each book a unique alphanumeric call number based on subject matter. Over 170 million items are catalogued using this system across participating institutions.",
    ],
    "emotional": [
        "BREAKING: A catastrophic explosion at a chemical plant has killed 47 workers and sent a toxic cloud over residential neighborhoods. Emergency services report hundreds of children hospitalized with severe respiratory distress. Families are desperately searching for missing loved ones amid the chaos.",
        "URGENT: Surveillance footage captures the moment a school bus carrying 38 children plunges off a bridge into freezing rapids below. Rescue divers describe horrifying scenes underwater. Parents collapse at the scene as body bags are carried from the riverbank.",
        "ALERT: A previously unknown pathogen has been detected spreading through the water supply of a major city, causing rapid organ failure within hours of exposure. Hospitals are overwhelmed. Officials warn the contamination may have reached neighboring regions before detection.",
    ],
    "task_relevant": [
        "Recent studies on volcanic monitoring have shown that micro-tremor frequency is an unreliable predictor of eruption risk. Seismologist Dr. Maria Fernandez argues that harmonic tremor patterns, rather than raw counts, provide the most actionable early warning data. Her 2024 paper demonstrated that tremor counts can fluctuate by 300 percent with no change in eruption probability.",
        "Deep-sea research expeditions frequently face challenges with equipment calibration at extreme pressures. A comprehensive review of 45 submarine missions found that sonar readings below 8,000 meters showed systematic errors of up to 15 percent in feature counts. Many reported discoveries of hydrothermal vents have been disputed after reanalysis.",
        "Radio telescope arrays often produce spurious signals that can mimic astrophysical phenomena. A landmark 2025 analysis revealed that 23 percent of reported fast radio bursts in the literature were actually interference from satellite constellations. Distance estimates based on dispersion measures carry uncertainties of plus or minus 40 percent for extragalactic sources.",
    ],
}

random.seed(789)
data = []
task_id = 0

for passage in PASSAGES:
    for change_type in ["minor", "major"]:
        changed_text = passage[f"{change_type}_change"]
        expected_detail = passage[f"{change_type}_detail"]

        for disruptor_type in ["neutral", "emotional", "task_relevant"]:
            disruptor = random.choice(DISRUPTORS[disruptor_type])

            prompt = (
                "Read Version A, then the interlude, then Version B carefully. "
                "Identify what factual detail changed between the two versions.\n\n"
                f"=== VERSION A ===\n{passage['original']}\n\n"
                f"=== INTERLUDE ===\n{disruptor}\n\n"
                f"=== VERSION B ===\n{changed_text}\n\n"
                "What specific factual detail changed between Version A and Version B? "
                "Ignore the interlude — it is unrelated to the change. "
                "State the change precisely: what was the original detail and what did it become?"
            )

            data.append({
                "task_id": task_id,
                "prompt": prompt,
                "expected": expected_detail,
                "passage_id": passage["id"],
                "change_type": change_type,
                "disruptor_type": disruptor_type,
            })
            task_id += 1

    # Control: no-change condition
    for disruptor_type in ["neutral", "emotional", "task_relevant"]:
        disruptor = random.choice(DISRUPTORS[disruptor_type])

        prompt = (
            "Read Version A, then the interlude, then Version B carefully. "
            "Identify what factual detail changed between the two versions. "
            "If nothing changed, respond with exactly: NO CHANGE\n\n"
            f"=== VERSION A ===\n{passage['original']}\n\n"
            f"=== INTERLUDE ===\n{disruptor}\n\n"
            f"=== VERSION B ===\n{passage['original']}\n\n"
            "What specific factual detail changed between Version A and Version B? "
            "Ignore the interlude. If the passages are identical, respond with exactly: NO CHANGE"
        )

        data.append({
            "task_id": task_id,
            "prompt": prompt,
            "expected": "NO CHANGE",
            "passage_id": passage["id"],
            "change_type": "none",
            "disruptor_type": disruptor_type,
        })
        task_id += 1

df_all = pd.DataFrame(data)
print(f"Generated {len(df_all)} items")
print(f"By change type: {dict(df_all['change_type'].value_counts())}")
print(f"By disruptor type: {dict(df_all['disruptor_type'].value_counts())}")

## Task Definition

In [ ]:
@kbench.task(
    name="mudsplash",
    description="Does salient/emotional disruptor content cause models to miss factual changes?"
)
def mudsplash(
    llm,
    prompt: str,
    expected: str,
    change_type: str,
    disruptor_type: str,
) -> bool:
    response = llm.prompt(prompt)
    response = strip_thinking(response)
    resp_norm = normalize(response)
    exp_norm = normalize(expected)

    if expected == "NO CHANGE":
        return "no change" in resp_norm

    key_terms = [normalize(t) for t in re.split(r"changed to|→|->|changed from", expected.lower()) if t.strip()]
    if len(key_terms) >= 2:
        return all(term in resp_norm for term in key_terms)
    else:
        return exp_norm in resp_norm

## Run Evaluation

Uses `kbench.llm` — the default model. Use Kaggle's **"Add Models"** button to run across multiple models.

In [ ]:
eval_df = df_all[["prompt", "expected", "change_type", "disruptor_type"]].copy()

print(f"Running {len(eval_df)} mudsplash items with kbench.llm...")
runs = mudsplash.evaluate(
    llm=[kbench.llm],
    evaluation_data=eval_df,
    n_jobs=2,
    timeout=120,
    max_attempts=2,
)
results = runs.as_dataframe()
results["correct"] = results["result"].apply(lambda r: float(r) if isinstance(r, bool) else float(r))
results = results.merge(
    df_all[["prompt", "passage_id", "change_type", "disruptor_type"]],
    on="prompt", how="left", suffixes=("", "_meta")
)
print(f"Collected {len(results)} results")

## Results & Analysis

In [ ]:
print("=== Detection Rate by Change Type × Disruptor Type ===")
pivot = results.groupby(["change_type", "disruptor_type"])["correct"].mean().unstack()
print(pivot.to_string(float_format="%.2f"))

print("\n=== Attentional Capture Effect ===")
for ct in ["minor", "major"]:
    ct_data = results[results["change_type"] == ct]
    neutral = ct_data[ct_data["disruptor_type"] == "neutral"]["correct"].mean()
    emotional = ct_data[ct_data["disruptor_type"] == "emotional"]["correct"].mean()
    task_rel = ct_data[ct_data["disruptor_type"] == "task_relevant"]["correct"].mean()
    print(f"{ct}: neutral={neutral:.2%}, emotional={emotional:.2%} (Δ={neutral-emotional:+.2%}), "
          f"task-relevant={task_rel:.2%} (Δ={neutral-task_rel:+.2%})")

# False alarm rate by disruptor type
no_change = results[results["change_type"] == "none"]
if len(no_change) > 0:
    print("\n=== False Alarm Rate by Disruptor Type ===")
    for dt in ["neutral", "emotional", "task_relevant"]:
        subset = no_change[no_change["disruptor_type"] == dt]
        fa = 1 - subset["correct"].mean() if len(subset) > 0 else 0
        print(f"  {dt}: {fa:.2%}")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "matplotlib"], check=True)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

disruptor_labels = ["neutral", "emotional", "task_relevant"]
x = np.arange(len(disruptor_labels))
width = 0.25
colors = {"minor": "orange", "major": "steelblue", "none": "tomato"}

for i, ct in enumerate(["minor", "major", "none"]):
    ct_data = results[results["change_type"] == ct]
    means = [ct_data[ct_data["disruptor_type"] == dt]["correct"].mean() for dt in disruptor_labels]
    if ct == "none":
        means = [1 - m for m in means]  # Show false alarm rate
        label = "False alarm rate"
    else:
        label = f"{ct} change detection"
    ax.bar(x + (i - 1) * width, means, width, label=label, color=colors[ct],
           edgecolor="black", linewidth=0.5)

ax.set_xlabel("Disruptor Type")
ax.set_ylabel("Rate")
ax.set_title("Mudsplash: Attentional Capture Effect on Change Detection")
ax.set_xticks(x)
ax.set_xticklabels(["Neutral\n(baseline)", "Emotional\n(shocking)", "Task-Relevant\n(confusing)"])
ax.set_ylim(0, 1.1)
ax.legend()
plt.tight_layout()
plt.savefig("mudsplash_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to mudsplash_results.png")